# Q19: Calibrating Classification Probabilities
**Topic:** Probability | **Difficulty:** Hard | **Time:** 15 min
**Sheet:** Grind75ML

---

## Question
What approach would you take to adjust (calibrate) the probabilities generated by a classification model for better accuracy?

## Detailed Answer

### What is Probability Calibration?
A **well-calibrated** model means: if it predicts 0.8 for 100 different samples, roughly 80 of them should actually be positive.

Many models (especially SVMs, Random Forests, and boosted trees) produce poorly calibrated probabilities.

### Method 1: Platt Scaling (Sigmoid Calibration)
Fit a **logistic regression** on the model's raw scores using a held-out validation set:

$$P(y=1|f) = \frac{1}{1 + \exp(Af + B)}$$

- Parameters A, B are learned on validation data
- Best for: models with sigmoid-shaped miscalibration
- Works well for SVMs, neural networks

### Method 2: Isotonic Regression
Fit a **non-parametric, monotonically non-decreasing** function:
- More flexible than Platt scaling
- Best for: large datasets, non-sigmoid miscalibration
- Risk of overfitting on small datasets

### Method 3: Temperature Scaling
Divide logits by a learned temperature $T$ before softmax:

$$P(y=k|x) = \frac{\exp(z_k / T)}{\sum_j \exp(z_j / T)}$$

- $T > 1$: softens probabilities (less confident)
- $T < 1$: sharpens probabilities (more confident)
- Best for: neural networks, multi-class problems
- Single parameter — low overfitting risk

### Evaluation: Reliability Diagram + ECE
- **Reliability Diagram**: Plot predicted probability vs actual frequency, binned
- **Expected Calibration Error (ECE)**: Weighted average of |predicted - actual| per bin

$$\text{ECE} = \sum_{b=1}^{B} \frac{n_b}{N} |\text{acc}(b) - \text{conf}(b)|$$

### Which models need calibration?
| Model | Calibration Quality |
|-------|--------------------|
| Logistic Regression | Good (inherently calibrated) |
| Neural Networks | Poor (overconfident, especially deep) |
| Random Forest | Poor (biased toward 0 and 1) |
| SVM | No probabilities natively |
| Gradient Boosting | Moderate |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

# Generate data
X, y = make_classification(n_samples=5000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Uncalibrated RF
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
prob_uncal = rf.predict_proba(X_test)[:, 1]

# Calibrated RF (Platt Scaling)
rf_platt = CalibratedClassifierCV(rf, method='sigmoid', cv=5)
rf_platt.fit(X_train, y_train)
prob_platt = rf_platt.predict_proba(X_test)[:, 1]

# Calibrated RF (Isotonic)
rf_iso = CalibratedClassifierCV(rf, method='isotonic', cv=5)
rf_iso.fit(X_train, y_train)
prob_iso = rf_iso.predict_proba(X_test)[:, 1]

# Plot reliability diagrams
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')

for prob, name, color in [(prob_uncal, 'Uncalibrated RF', 'red'),
                           (prob_platt, 'Platt Scaling', 'blue'),
                           (prob_iso, 'Isotonic', 'green')]:
    frac_pos, mean_pred = calibration_curve(y_test, prob, n_bins=10)
    brier = brier_score_loss(y_test, prob)
    ax.plot(mean_pred, frac_pos, 's-', color=color, label=f'{name} (Brier={brier:.4f})')

ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Reliability Diagram')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways
- **Always calibrate** if probabilities are used for decisions (thresholding, ranking, risk assessment)
- **Temperature scaling** is the go-to for neural networks (simple, effective)
- **Platt scaling** for small datasets; **Isotonic** for large datasets
- Use **Brier score** and **reliability diagrams** to evaluate calibration quality